# Real-Time Financial Fraud Detection Engine & Feature Drift Monitoring Pipeline

This notebook contains the complete production-ready Python pipeline for SentinelShield. It demonstrates:
1. Ingesting transaction streams from **Apache Kafka** using **PySpark Structured Streaming**.
2. Training an **Isolation Forest** model to detect anomalies in high-dimensional financial datasets.
3. Over-sampling rare fraud events using **SMOTE** to handle extreme class imbalance (99% normal, 1% fraud).
4. Conducting **Kolmogorov-Smirnov statistical tests** to track feature drift in production data.

In [ ]:
# 1. Environment Setup & Library Imports
import os
import sys
import numpy as np
import pandas as pd
from scipy import stats

# PySpark ML & Streaming
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, udf
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType

# Modeling & Imbalance Processing
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, average_precision_score, precision_recall_curve
from imblearn.over_sampling import SMOTE

# Drift Tracking
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset

## 2. Ingesting Streaming Transactions with Spark & Kafka

Here we configure Spark to subscribe to a Kafka topic containing live transactional events.

In [ ]:
# Initialize PySpark with Kafka dependencies
spark = SparkSession.builder \
    .appName("SentinelShield-Streaming") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0") \
    .getOrCreate()

# Define the schema of incoming transaction JSON objects
transaction_schema = StructType([
    StructField("transaction_id", StringType(), True),
    StructField("card_id", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("velocity_1h", DoubleType(), True),  # transactions in last hour
    StructField("distance_from_home", DoubleType(), True),
    StructField("is_international", LongType(), True)
])

# Read stream from local Kafka broker
kafka_stream_df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribe", "financial-transactions") \
    .option("startingOffsets", "latest") \
    .load()

# Parse raw binary payload into structured columns
transactions_df = kafka_stream_df \
    .selectExpr("CAST(value AS STRING) as json_payload") \
    .select(from_json("json_payload", transaction_schema).alias("data")) \
    .select("data.*")

## 3. Simulating Baseline Training Data (Extreme Class Imbalance)

In credit card transactions, fraud represents less than 0.2% of all entries. We simulate a baseline dataset with a 99:1 normal-to-fraud ratio.

In [ ]:
np.random.seed(42)
n_samples = 100000
n_fraud = 1000  # 1% fraud rate

# Generate Normal Transactions
normal_amount = np.random.normal(loc=45.0, scale=25.0, size=n_samples - n_fraud)
normal_velocity = np.random.poisson(lam=1.5, size=n_samples - n_fraud)
normal_distance = np.random.exponential(scale=8.0, size=n_samples - n_fraud)

# Generate Fraudulent Transactions (Higher amounts, velocities, and distances)
fraud_amount = np.random.normal(loc=450.0, scale=150.0, size=n_fraud)
fraud_velocity = np.random.poisson(lam=6.0, size=n_fraud)
fraud_distance = np.random.exponential(scale=75.0, size=n_fraud)

# Consolidate into DataFrame
data = pd.DataFrame({
    'amount': np.concatenate([normal_amount, fraud_amount]),
    'velocity_1h': np.concatenate([normal_velocity, fraud_velocity]),
    'distance_from_home': np.concatenate([normal_distance, fraud_distance]),
    'label': np.concatenate([np.zeros(n_samples - n_fraud), np.ones(n_fraud)])
})

# Clean negative amounts
data['amount'] = data['amount'].clip(lower=1.0)

print(f"Class Distribution:\n{data['label'].value_value_counts(normalize=True) * 100} %")

## 4. SMOTE (Synthetic Minority Over-sampling Technique) for Class Balancing

Standard algorithms degrade when trained on imbalanced data. We apply SMOTE to balance the training set.

In [ ]:
X = data.drop(columns=['label'])
y = data['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Original training shape: {X_train.shape}")

# Apply SMOTE
smote = SMOTE(sampling_strategy=0.1, random_state=42)  # Boost fraud up to 10% of majority class
X_resampled, y_resampled = smote.fit_resample(X_train, y_train)

print(f"SMOTE resampled training shape: {X_resampled.shape}")

## 5. Model Training: Isolation Forest (Anomaly Detection)

Isolation Forest builds isolated search paths. Shorter path lengths correspond to outlier/fraud instances.

In [ ]:
# Train Isolation Forest on normal instances
X_train_normal = X_train[y_train == 0]
iso_forest = IsolationForest(n_estimators=100, contamination=0.01, random_state=42)
iso_forest.fit(X_train_normal)

# Evaluate anomaly scores on test set
# Isolation Forest score outputs are normalized (higher score means more anomalous)
scores = -iso_forest.decision_function(X_test)

# Map scores to binary flags using 99th percentile threshold
threshold = np.percentile(scores, 99)
y_pred = (scores > threshold).astype(int)

print("Classification Performance Report:")
print(classification_report(y_test, y_pred))

## 6. Real-Time Feature Drift Tracking (Evidently AI & KS-Test)

In production, user behavior changes (e.g. holiday shopping increases average transaction value). We track drift to prompt model retraining.

In [ ]:
# Simulate standard production data (no drift)
prod_amount_no_drift = np.random.normal(loc=45.0, scale=25.0, size=500).clip(lower=1.0)

# Simulate drifted production data (sudden spike in transaction amounts)
prod_amount_drifted = np.random.normal(loc=120.0, scale=50.0, size=500).clip(lower=1.0)

# Perform Kolmogorov-Smirnov Test (compares distributions)
ks_stat_no_drift, p_val_no_drift = stats.ks_2samp(X_train['amount'], prod_amount_no_drift)
ks_stat_drifted, p_val_drifted = stats.ks_2samp(X_train['amount'], prod_amount_drifted)

print(f"KS-Test (No Drift Scenario) -> p-value: {p_val_no_drift:.5f} (Drift Detected: {p_val_no_drift < 0.05})")
print(f"KS-Test (Drifted Scenario) -> p-value: {p_val_drifted:.5f} (Drift Detected: {p_val_drifted < 0.05})")

# Generate evidently drift dashboard data
reference_df = pd.DataFrame({'amount': X_train['amount']})
current_df = pd.DataFrame({'amount': prod_amount_drifted})

data_drift_report = Report(metrics=[DataDriftPreset()])
data_drift_report.run(reference_data=reference_df, current_data=current_df)
data_drift_report.save_html("data_drift_report.html")
print("Evidently AI Drift Report saved successfully as 'data_drift_report.html'.")